# Hour 3 · Finding formulas with n-grams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/3a_ngrams_formulas.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F3a_ngrams_formulas.ipynb)


**Goal:** find the fixed expressions (formulas) that pervade the corpus — letter greetings, divine epithets, ritual instructions.

*Reading:* `docs/05-formulas.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

## 1. n-gram helper

In [ ]:
def ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
ngrams(["tḥm","mlk","l","mlkt","rgm"], 2)

## 2. Most frequent bigrams and trigrams across the corpus

In [ ]:
from collections import Counter
bi, tri = Counter(), Counter()
for t in texts:
    bi.update(ngrams(t["tokens"],2))
    tri.update(ngrams(t["tokens"],3))
print("Top bigrams:")
[print(f"  {c:3d}  {g}") for g,c in bi.most_common(12)]
print("\nTop trigrams:")
[print(f"  {c:3d}  {g}") for g,c in tri.most_common(12)];

## 3. Formulas differ by genre

In [ ]:
from data.loader import texts_by_genre
for g, items in texts_by_genre(texts).items():
    c = Counter()
    for t in items: c.update(ngrams(t["tokens"],2))
    print(f"{g:20s}: {[x for x,_ in c.most_common(6)]}")

## 4. Hunt a specific formula

In [ ]:
targets = ["yšlm lk", "l pʿn", "kṯr w ḫss"]
for q in targets:
    n = sum(q in " ".join(x["tokens"]) for x in texts)
    print(f"{q!r:14s} in {n} tablets")

## 5. Discussion
- Frequent n-grams are candidate **formulas** for closer study (e.g. *kṯr w ḫss* = the god Kothar-wa-Hasis; *rbt aṯrt ym* = 'Lady Athirat of the Sea').
- **TODO:** filter trivial particles (`w`, `l`, `b`); attach KTU references.

> Your CUC SQL-console queries (frequent n-grams *with references*) do this on the full word-level dataset — see `docs/00-resources.md`.